In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize

In [2]:

def opt_min_var(prices: pd.DataFrame):
    rets = prices.pct_change().dropna()
    cov = rets.cov()
    var = lambda w: w.T @ cov @ w
    n_assets = len(prices.columns)
    w_0 = np.ones(n_assets) / n_assets
    restr = lambda w: sum(w) - 1
    rangos = [(0, 1)] * n_assets
    tol = 1e-32
    results = minimize(
        fun=var,
        x0=w_0,
        bounds=rangos,
        constraints={'fun': restr, 'type': 'eq'},
        tol=tol)

    return results.x
# MAX SHARPE

def opt_max_sharpe(price: pd.DataFrame, rf: float):
    rets = price.pct_change().dropna()
    cov = rets.cov()
    mu = rets.mean()
    sharpe = lambda w: -((sum(w*mu)-rf)/np.sqrt(w.T@cov@w))
    restr = lambda w: sum(w) - 1
    n_assets = len(price.columns)
    w_0 = np.ones(n_assets) / n_assets
    tol = 1e-32
    rangos = [(0,1)] * n_assets
    results = minimize(fun=sharpe,x0=w_0,bounds=rangos,constraints={'fun': restr, 'type': 'eq'},tol=tol)

    return results.x

# SEMIVAR TARJET
def opt_target_semivar(prices: pd.DataFrame, benchmark: pd.Series):
    rets=prices.pct_change().dropna()
    rets_benchmark=benchmark.pct_change().dropna()
    alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()
    below_zero_t=(alpha[alpha<0]).fillna(0)
    downside_risk_t=np.array(below_zero_t.std())
    corr=alpha.corr()
    n_assets=len(prices.columns)
    semivar_matrix=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*corr

    func_target=lambda w: w.T @ semivar_matrix @ w
    restr=lambda w: sum(w)-1
    w_0=np.ones(n_assets)/n_assets
    tol=1e-12
    rangos=[(0,1)]*n_assets
    
    results=minimize(fun=func_target,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    if not results.success:
        print(results.message)

    return results.x
# BACKTESTING
def static_backtesting(prices: pd.DataFrame, benchmark: pd.DataFrame, weights: dict, capital: float):

    rets=prices.pct_change().dropna()
    rets_b=benchmark.pct_change().dropna()
    evolucion=pd.DataFrame(index=rets.index)
    for strategy in weights.keys():

        if strategy =='Benchmark':
            evol_temp=rets_b+1
            evol_temp.iloc[0]=capital
            evol_temp=evol_temp.cumprod()
            evolucion[strategy]=evol_temp.values
        else:
            R_p=np.sum(rets * weights[strategy], axis=1)
            evol_temp=R_p+1
            evol_temp.iloc[0]=capital
            evol_temp=evol_temp.cumprod()
            evolucion[strategy]=evol_temp.values
    plt.Figure(figsize=(12, 6))
    plt.plot(evolucion,label=evolucion.columns)
    plt.title('Backtesting')
    plt.xlabel('Date')
    plt.ylabel('Portafolio Value')
    plt.legend(evolucion.columns)
    plt.grid()

    return evolucion

# 1.- Función de Métricas de Desempeño. (20%)

In [3]:

def performance_metrics(history, rf):

    rets = history.pct_change().dropna()

    metrics = pd.DataFrame(columns=history.columns)

    metrics.loc['Expected Return'] = rets.mean()*252

    metrics.loc['Volatility'] = rets.std()*np.sqrt(252)

    metrics.loc['Sharpe Ratio'] = (
        (metrics.loc['Expected Return']-rf) /
        metrics.loc['Volatility']
    )

    metrics.loc['Downside Risk'] = (
        rets[rets<0].fillna(0).std()*np.sqrt(252)
    )

    metrics.loc['Upside Risk'] = (
        rets[rets>0].fillna(0).std()*np.sqrt(252)
    )

    metrics.loc['Omega'] = (
        metrics.loc['Upside Risk'] /
        metrics.loc['Downside Risk']
    )

    metrics.loc['Beta'] = (
        rets.cov()['Benchmark'] /
        rets['Benchmark'].var()
    )

    metrics.loc['Jensen Alpha'] = (
        metrics.loc['Expected Return'] -
        (rf + metrics.loc['Beta'] *
        (metrics.loc['Expected Return']['Benchmark']-rf))
    )

    metrics.loc['Treynor'] = (
        (metrics.loc['Expected Return']-rf) /
        metrics.loc['Beta']
    )

    metrics.loc['Sortino'] = (
        (metrics.loc['Expected Return']-rf) /
        metrics.loc['Downside Risk']
    )

    return metrics



# 2.- Backtesting de Portafolio. (50%)

In [4]:
tickers = ["LIVEPOLC-1.MX", "KOFUBL.MX", "CEMEXCPO.MX", "CHDRAUIB.MX", "CUERVO.MX", "LABB.MX", "RA.MX"]
opt_prices=yf.download(tickers, start='2022-10-21', end='2024-10-21', progress=False)['Close']
bt=yf.download(tickers + ['^MXX'], start='2024-10-21', end='2026-10-21', progress=False)['Close'].dropna()
bt_prices=bt[opt_prices.columns]
benchmark=bt['^MXX']


/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_17783/4226632162.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  opt_prices=yf.download(tickers, start='2022-10-21', end='2024-10-21', progress=False)['Close']
/var/folders/x9/60hcc2ln62d3r3xyv90d3wl40000gn/T/ipykernel_17783/4226632162.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  bt=yf.download(tickers + ['^MXX'], start='2024-10-21', end='2026-10-21', progress=False)['Close'].dropna()


In [6]:
opt_returns=opt_prices.pct_change().dropna()
w_min_var=opt_min_var(opt_returns)
w_max_sharpe=opt_max_sharpe(opt_returns, 0.065)
strategies={'Min Var': w_min_var, 'Max Sharpe': w_max_sharpe,'Benchmark': 1,'Omega':w_omega,'Semivar':w_tarjet_semivar}
capital=1000000
evol=static_backtesting(bt_prices, benchmark, strategies, capital)
evol

/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:136: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2842: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:136: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2842: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


NameError: name 'w_omega' is not defined

# 3.- Selección de Estrategia de Inversión. (20%)

# 4.- Calculo de Pesos para Portafolio Real. (10%)